# Ceftazidime x *E. coli* — Aggregated Iterative (4 runs)

**Data:** Pooled A+B+C+D, 10% fixed test set extracted upfront.

**4 sequential runs** with different 85/15 split seeds, each model evolving:

| Model | Run 1 | Runs 2–4 |
|---|---|---|
| **LR** | GridSearchCV(C) + CV threshold | Independent (same procedure, new split) |
| **MLP** | 6×6 grid lr×dropout + threshold | Warm-start from previous weights, search only LR (6 values) |
| **RF** | GridSearchCV + CV threshold | `warm_start=True`, accumulates trees |

**Mixed seed = new 85/15 split seed each run.** All runs evaluate on the same fixed 10% test set.

**Output:** per-run tracking table, progression line chart, final heatmap (mean ± std across 4 runs).

In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings, copy, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (GridSearchCV, train_test_split, cross_val_predict)
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

SEEDS = [42, 123, 456, 789]
TEST_SEED = 99
print(f"Run seeds: {SEEDS}  |  Test split seed: {TEST_SEED}")

In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

OUT_DIR = Path("./results_iterative")
OUT_DIR.mkdir(exist_ok=True)

SITES_PATHS = {
    "A": DRYAD / "Processed/Proc_DRIAMS-A" / "Ceftazidime" / "data.csv",
    "B": DRYAD / "Processed/Proc_DRIAMS-B" / "Ceftazidime" / "data.csv",
    "C": DRYAD / "Processed/Proc_DRIAMS-C" / "Ceftazidime" / "data.csv",
    "D": DRYAD / "Processed/Proc_DRIAMS-D" / "Ceftazidime" / "data.csv",
}

In [ ]:
# ── Load Ceftazidime + E. coli from all 4 sites ──
SPECIES = "Escherichia coli"
site_data = {}

for site, path in SITES_PATHS.items():
    df = pd.read_csv(path)
    df_eco = df[df["species"] == SPECIES].copy()
    bin_cols = [c for c in df_eco.columns if c.startswith("bin_")]
    X = df_eco[bin_cols].to_numpy(dtype="float32")
    y = df_eco["label"].to_numpy(dtype="int64")
    site_data[site] = {"X": X, "y": y, "n": len(y)}
    n_r, n_s = (y == 1).sum(), (y == 0).sum()
    print(f"  Site {site}: {len(y)} samples ({n_s} S, {n_r} R, {n_r/len(y)*100:.1f}% R)")

X_all = np.concatenate([site_data[s]["X"] for s in "ABCD"])
y_all = np.concatenate([site_data[s]["y"] for s in "ABCD"])
print(f"\nTotal pooled: {len(X_all)} samples")

In [ ]:
# ── Fixed 10% test set (NEVER used during training) ──
X_trainval_raw, X_test_raw, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=0.10, stratify=y_all, random_state=TEST_SEED)

print(f"Fixed test:  {len(X_test_raw)} samples  ({len(X_test_raw)/len(X_all)*100:.1f}%)")
print(f"Trainval pool: {len(X_trainval_raw)} samples  ({len(X_trainval_raw)/len(X_all)*100:.1f}%)")
print(f"  Resistant: test={(y_test==1).sum()}, trainval={(y_trainval==1).sum()}")

In [ ]:
# ── Preprocessing: fit on 90% trainval pool, apply to fixed test once ──
state_all = fit_input_transform(X_trainval_raw, "log1p+standardize")
X_trainval_pp = apply_input_transform(X_trainval_raw, state_all)
X_test_pp = apply_input_transform(X_test_raw, state_all)
print("Preprocessing fitted on trainval pool, applied to fixed test.")
print(f"  Trainval shape: {X_trainval_pp.shape}")
print(f"  Test shape:     {X_test_pp.shape}")

In [ ]:
# ── Shared tuning constants ──
C_GRID = np.linspace(5e-5, 1e-3, 15)
LR_GRID  = np.linspace(1e-4, 5e-4, 6)
DROP_GRID = np.linspace(0.2, 0.6, 6)
THRESHOLDS = np.linspace(0.05, 0.95, 91)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RF_PARAM_GRID = {
    "n_estimators": [100, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_leaf": [2, 5, 10],
    "class_weight": ["balanced", "balanced_subsample"],
}

print(f"Device: {DEVICE}")
print(f"C grid: {len(C_GRID)}  |  MLP grid: {len(LR_GRID)}x{len(DROP_GRID)}={len(LR_GRID)*len(DROP_GRID)}")
print(f"RF grid: {np.prod([len(v) for v in RF_PARAM_GRID.values()])} combos")

In [ ]:
# ── Simple PyTorch Dataset for binary classification ──
class BinDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CUSTOM BINARY MLP  (warm-start support via load_state_dict / get_state_dict)
# ═══════════════════════════════════════════════════════════════════════════

class BinaryMaldiMLP:
    def __init__(self, hidden_dim=512, head_dims=(256, 128), use_attention=False,
                 dropout_high=0.3, dropout_low=0.2, device=DEVICE):
        self.hidden_dim = hidden_dim
        self.head_dims = head_dims
        self.use_attention = use_attention
        self.dropout_high = dropout_high
        self.dropout_low = dropout_low
        self.device = device

    def _build(self):
        return SpectralAttentionMLP(
            input_dim=6000, n_classes=2,
            hidden_dim=self.hidden_dim, head_dims=self.head_dims,
            use_attention=self.use_attention,
            dropout_high=self.dropout_high, dropout_low=self.dropout_low)

    def get_state_dict(self):
        return {k: v.cpu().clone() for k, v in self.model_.state_dict().items()}

    def load_state_dict(self, sd):
        self.model_ = self._build().to(self.device)
        self.model_.load_state_dict(sd)
        self.model_.eval()

    def fit(self, X_tr, y_tr, X_val=None, y_val=None,
            lr=1e-4, weight_decay=1e-3, batch_size=64, epochs=50,
            warmup_epochs=0, early_stopping_patience=10,
            verbose=False):
        """Train from scratch (self.model_ is None) or warm-start (self.model_ exists)."""
        if not hasattr(self, 'model_') or self.model_ is None:
            self.model_ = self._build().to(self.device)

        ds = BinDataset(X_tr, y_tr)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
        val_dl = None
        if X_val is not None and y_val is not None:
            val_dl = DataLoader(BinDataset(X_val, y_val), batch_size=batch_size*2, shuffle=False)

        opt = torch.optim.AdamW(self.model_.parameters(), lr=lr, weight_decay=weight_decay)
        warmup = max(0, warmup_epochs)
        t_max = max(1, epochs - warmup)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=t_max, eta_min=1e-6)
        crit = nn.CrossEntropyLoss()

        best_val, best_sd, patience = float("inf"), None, 0
        for ep in range(epochs):
            self.model_.train()
            for xb, yb in dl:
                xb, yb = xb.to(self.device), yb.to(self.device)
                if ep < warmup:
                    for pg in opt.param_groups:
                        pg["lr"] = lr * (ep + 1) / warmup
                opt.zero_grad()
                loss = crit(self.model_(xb), yb)
                loss.backward(); opt.step()
            if ep >= warmup: sched.step()

            if val_dl is not None:
                self.model_.eval(); vl = 0.0
                with torch.no_grad():
                    for xb, yb in val_dl:
                        xb, yb = xb.to(self.device), yb.to(self.device)
                        vl += crit(self.model_(xb), yb).item()
                vl /= len(val_dl)
                if vl < best_val:
                    best_val = vl; best_sd = self.get_state_dict(); patience = 0
                else:
                    patience += 1
                    if patience >= early_stopping_patience:
                        if verbose: print(f"    Early stop @ ep {ep+1}")
                        break

        if best_sd is not None:
            self.model_.load_state_dict(best_sd)
        self.model_.eval()

    def predict_proba(self, X):
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        self.model_.eval()
        with torch.no_grad():
            return F.softmax(self.model_(X_t), dim=1).cpu().numpy()[:, 1]

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MLP grid search  (6x6 lr x dropout) — returns best model + params
# ═══════════════════════════════════════════════════════════════════════════

def grid_search_mlp(X_train, y_train, device=DEVICE):
    """Full 6x6 grid. Returns (BinaryMaldiMLP, best_lr, best_dh, best_threshold)."""
    X_st, X_sv, y_st, y_sv = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=SEEDS[0])

    best_ba, best_lr, best_dh, best_t = -1.0, None, None, 0.5
    for lr_val in LR_GRID:
        for d in DROP_GRID:
            dh, dl = d, d / 2.0
            m = BinaryMaldiMLP(hidden_dim=512, dropout_high=dh, dropout_low=dl, device=device)
            m.fit(X_st, y_st, X_sv, y_sv, lr=lr_val, weight_decay=1e-3,
                  batch_size=64, epochs=50, early_stopping_patience=10, verbose=False)
            proba = m.predict_proba(X_sv)
            for t in THRESHOLDS:
                ba = balanced_accuracy_score(y_sv, proba >= t)
                if ba > best_ba:
                    best_ba = ba; best_lr = lr_val; best_dh = dh; best_t = t
        print(f"  lr={lr_val:.1e}  best-drop={best_dh:.1f}  BA={best_ba:.4f}")

    print(f"  Best: lr={best_lr:.1e}  dropout={best_dh:.1f}  threshold={best_t:.3f}")

    # Retrain best on full train
    m_final = BinaryMaldiMLP(hidden_dim=512, dropout_high=best_dh, dropout_low=best_dh/2.0, device=device)
    m_final.fit(X_train, y_train, None, None, lr=best_lr, weight_decay=1e-4,
                batch_size=64, epochs=100, warmup_epochs=10, early_stopping_patience=15, verbose=True)
    return m_final, best_lr, best_dh, best_t

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# MLP fine-tune  (LR-only search, warm-start from previous weights)
# ═══════════════════════════════════════════════════════════════════════════

def fine_tune_mlp(X_train, y_train, prev_state_dict, dropout_high, prev_best_lr, device=DEVICE, seed=42):
    """Warm-start: search LR only (6 values narrower than original grid), fine-tune."""
    X_st, X_sv, y_st, y_sv = train_test_split(
        X_train, y_train, test_size=0.15, stratify=y_train, random_state=seed)

    ft_lr_grid = np.linspace(prev_best_lr / 5.0, prev_best_lr, 6)
    best_ba, best_lr, best_t = -1.0, None, 0.5
    for lr_val in ft_lr_grid:
        m = BinaryMaldiMLP(hidden_dim=512, dropout_high=dropout_high,
                           dropout_low=dropout_high / 2.0, device=device)
        m.load_state_dict(prev_state_dict)
        m.fit(X_st, y_st, X_sv, y_sv, lr=lr_val, weight_decay=1e-4,
              batch_size=64, epochs=50, early_stopping_patience=10, verbose=False)
        proba = m.predict_proba(X_sv)
        for t in THRESHOLDS:
            ba = balanced_accuracy_score(y_sv, proba >= t)
            if ba > best_ba:
                best_ba = ba; best_lr = lr_val; best_t = t
        print(f"  lr={lr_val:.1e}  BA={balanced_accuracy_score(y_sv, proba >= 0.5):.4f}")

    print(f"  Best FT-lr={best_lr:.1e}  threshold={best_t:.3f}")

    # Fine-tune on full train
    m_final = BinaryMaldiMLP(hidden_dim=512, dropout_high=dropout_high,
                             dropout_low=dropout_high / 2.0, device=device)
    m_final.load_state_dict(prev_state_dict)
    m_final.fit(X_train, y_train, None, None, lr=best_lr, weight_decay=1e-4,
                batch_size=64, epochs=100, warmup_epochs=10, early_stopping_patience=15, verbose=True)
    return m_final, best_lr, best_t

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LR  (independent GridSearchCV each run)
# ═══════════════════════════════════════════════════════════════════════════

def train_lr_run(X_train, y_train):
    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced",
                           max_iter=5000, random_state=SEED),
        param_grid={"C": C_GRID}, cv=3, scoring="balanced_accuracy", n_jobs=-1)
    grid.fit(X_train, y_train)
    best_c = grid.best_params_["C"]

    cv_proba = cross_val_predict(
        LogisticRegression(C=best_c, penalty="l2", solver="lbfgs",
                           class_weight="balanced", max_iter=5000, random_state=SEED),
        X_train, y_train, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    best_t = THRESHOLDS[np.argmax(
        [balanced_accuracy_score(y_train, cv_proba >= t) for t in THRESHOLDS])]

    lr = LogisticRegression(C=best_c, penalty="l2", solver="lbfgs",
                            class_weight="balanced", max_iter=5000, random_state=SEED)
    lr.fit(X_train, y_train)
    return lr, best_c, best_t

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RF  (GridSearchCV run 1; warm_start runs 2–4)
# ═══════════════════════════════════════════════════════════════════════════

def train_rf_run(X_train, y_train, prev_rf=None, n_est_per_run=None):
    """If prev_rf is None: full GridSearchCV, return (rf, threshold, n_est_per_run).
    Else: warm_start on same rf object, add n_est_per_run trees, return (rf, threshold, n_est_per_run)."""
    if prev_rf is None:
        grid = GridSearchCV(
            RandomForestClassifier(oob_score=True, random_state=SEED, n_jobs=-1),
            param_grid=RF_PARAM_GRID, cv=3, scoring="balanced_accuracy", n_jobs=-1)
        grid.fit(X_train, y_train)
        rf = grid.best_estimator_
        n_est_per_run = rf.n_estimators
        rf.set_params(warm_start=True)  # prepare for accumulation
        print(f"  Best: {grid.best_params_}  ({n_est_per_run} trees/run)")
    else:
        rf = prev_rf
        current_total = rf.n_estimators
        rf.set_params(n_estimators=current_total + n_est_per_run)
        rf.fit(X_train, y_train)
        print(f"  Warm-start: now {rf.n_estimators} trees total  (+{n_est_per_run})")

    # CV threshold: use a FRESH clone with warm_start=False (same n_estimators)
    cv_rf = RandomForestClassifier(
        n_estimators=rf.n_estimators, max_depth=rf.max_depth,
        min_samples_leaf=rf.min_samples_leaf, class_weight=rf.class_weight,
        random_state=SEED, n_jobs=-1, oob_score=False)
    cv_proba = cross_val_predict(
        cv_rf, X_train, y_train, cv=3, method="predict_proba", n_jobs=-1)[:, 1]
    best_t = THRESHOLDS[np.argmax(
        [balanced_accuracy_score(y_train, cv_proba >= t) for t in THRESHOLDS])]
    return rf, best_t, n_est_per_run

In [ ]:
# ── Evaluate on fixed test set ──
def eval_lr(lr, threshold, X_test, y_test):
    proba = lr.predict_proba(X_test)[:, 1]
    preds = proba >= threshold
    return balanced_accuracy_score(y_test, preds), roc_auc_score(y_test, proba)

def eval_mlp(mlp, threshold, X_test, y_test):
    proba = mlp.predict_proba(X_test)
    preds = proba >= threshold
    return balanced_accuracy_score(y_test, preds), roc_auc_score(y_test, proba)

def eval_rf(rf, threshold, X_test, y_test):
    proba = rf.predict_proba(X_test)[:, 1]
    preds = proba >= threshold
    return balanced_accuracy_score(y_test, preds), roc_auc_score(y_test, proba)

---
## Iterative Runs

In [ ]:
# ── Run 4 iterations ──
records = []  # list of dicts

# State trackers
prev_mlp_sd, prev_mlp_dh, prev_mlp_lr = None, None, None
prev_rf = None
n_est_per_run = None

for run_idx, run_seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"  RUN {run_idx+1}/4   seed={run_seed}   ({len(X_trainval_pp)} trainval, {len(X_test_pp)} test)")
    print(f"{'='*60}")

    # Split trainval into 85/15
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_trainval_pp, y_trainval, test_size=0.15, stratify=y_trainval,
        random_state=run_seed)
    print(f"  Split: train={len(X_tr)}  val={len(X_va)}  test={len(X_test_pp)}")

    # ── LR ──
    print("\n  --- LR ---")
    lr, best_c, lr_t = train_lr_run(X_tr, y_tr)
    lr_ba, lr_auc = eval_lr(lr, lr_t, X_test_pp, y_test)
    records.append({"Run": run_idx+1, "Seed": run_seed, "Model": "LR",
                    "Best_Param": f"C={best_c:.2e}", "Threshold": f"{lr_t:.3f}",
                    "BalAcc": lr_ba, "AUC": lr_auc,
                    "Extra": ""})
    print(f"  Test: BalAcc={lr_ba:.4f}  AUC={lr_auc:.4f}")

    # ── MLP ──
    print("\n  --- MLP ---")
    if run_idx == 0:
        mlp, mlp_lr, mlp_dh, mlp_t = grid_search_mlp(X_tr, y_tr)
        prev_mlp_sd = mlp.get_state_dict()
        prev_mlp_dh = mlp_dh
        prev_mlp_lr = mlp_lr
        extra = f"lr={mlp_lr:.1e} drop={mlp_dh:.1f}"
    else:
        mlp, mlp_lr, mlp_t = fine_tune_mlp(X_tr, y_tr, prev_mlp_sd, prev_mlp_dh,
                                            prev_mlp_lr, seed=run_seed)
        prev_mlp_sd = mlp.get_state_dict()
        prev_mlp_lr = mlp_lr
        extra = f"lr={mlp_lr:.1e} (warm)"

    mlp_ba, mlp_auc = eval_mlp(mlp, mlp_t, X_test_pp, y_test)
    records.append({"Run": run_idx+1, "Seed": run_seed, "Model": "MLP",
                    "Best_Param": f"lr={mlp_lr:.1e}", "Threshold": f"{mlp_t:.3f}",
                    "BalAcc": mlp_ba, "AUC": mlp_auc, "Extra": extra})
    print(f"  Test: BalAcc={mlp_ba:.4f}  AUC={mlp_auc:.4f}")

    # ── RF ──
    print("\n  --- RF ---")
    rf, rf_t, n_est_per_run = train_rf_run(X_tr, y_tr, prev_rf, n_est_per_run)
    rf_ba, rf_auc = eval_rf(rf, rf_t, X_test_pp, y_test)
    records.append({"Run": run_idx+1, "Seed": run_seed, "Model": "RF",
                    "Best_Param": f"{rf.n_estimators} trees", "Threshold": f"{rf_t:.3f}",
                    "BalAcc": rf_ba, "AUC": rf_auc,
                    "Extra": "warm" if run_idx > 0 else ""})
    prev_rf = rf
    print(f"  Test: BalAcc={rf_ba:.4f}  AUC={rf_auc:.4f}")

print("\nAll runs complete.")

---
## Per-Run Tracking

In [ ]:
# ── Per-run table ──
df_records = pd.DataFrame(records)
df_records_display = df_records.copy()
for col in ["BalAcc", "AUC"]:
    df_records_display[col] = df_records_display[col].apply(lambda x: f"{x:.4f}")
print(df_records_display.to_string(index=False))

# Save
df_records.to_csv(OUT_DIR / "per_run_results.csv", index=False)

In [ ]:
# ── Progression line chart ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for model, color, marker in [("LR", "#1f77b4", "o"), ("MLP", "#ff7f0e", "s"), ("RF", "#2ca02c", "^")]:
    sub = df_records[df_records["Model"] == model]
    ax1.plot(sub["Run"], sub["BalAcc"], marker=marker, color=color, label=model, linewidth=2, markersize=8)
    ax2.plot(sub["Run"], sub["AUC"], marker=marker, color=color, label=model, linewidth=2, markersize=8)

for ax in (ax1, ax2):
    ax.set_xlabel("Run"); ax.set_xticks([1, 2, 3, 4]); ax.legend(fontsize=10)
    ax.grid(True, ls='--', lw=0.5, color='gray', alpha=0.5)

ax1.set_ylabel("Balanced Accuracy"); ax1.set_title("Balanced Accuracy Progression")
ax2.set_ylabel("AUC-ROC"); ax2.set_title("AUC-ROC Progression")
fig.suptitle("Ceftazidime x E. coli — 4-Run Iterative Progression", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "progression.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── Final heatmap: mean ± std across runs ──
summary = df_records.groupby("Model").agg(BalAcc_mean=("BalAcc", "mean"),
                                          BalAcc_std=("BalAcc", "std"),
                                          AUC_mean=("AUC", "mean"),
                                          AUC_std=("AUC", "std")).reset_index()

df_ba_hm = pd.DataFrame({
    "Aggregated": summary.apply(lambda r: f"{r['BalAcc_mean']:.3f}\n±{r['BalAcc_std']:.3f}", axis=1).values
}, index=summary["Model"])
df_auc_hm = pd.DataFrame({
    "Aggregated": summary.apply(lambda r: f"{r['AUC_mean']:.3f}\n±{r['AUC_std']:.3f}", axis=1).values
}, index=summary["Model"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
sns.heatmap(summary[["BalAcc_mean"]].set_index(summary["Model"]).T,
            annot=df_ba_hm.T, fmt="", cmap="RdYlGn", vmin=0.5, vmax=1.0,
            linewidths=1.0, linecolor="white",
            cbar_kws={"label": "Balanced Accuracy", "shrink": 0.8}, ax=ax1)
ax1.set_title("Balanced Accuracy\n(mean ± std)", fontsize=11)
ax1.set_xlabel(""); ax1.set_ylabel("")

sns.heatmap(summary[["AUC_mean"]].set_index(summary["Model"]).T,
            annot=df_auc_hm.T, fmt="", cmap="RdYlGn", vmin=0.5, vmax=1.0,
            linewidths=1.0, linecolor="white",
            cbar_kws={"label": "AUC-ROC", "shrink": 0.8}, ax=ax2)
ax2.set_title("AUC-ROC\n(mean ± std)", fontsize=11)
ax2.set_xlabel(""); ax2.set_ylabel("")

fig.suptitle("Ceftazidime x E. coli — Aggregated Iterative (4 runs)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUT_DIR / "final_heatmap.pdf", bbox_inches="tight")
plt.show()

# Print summary
print("\nFinal summary (mean ± std across 4 runs):")
for _, r in summary.iterrows():
    print(f"  {r['Model']:5s}  BalAcc = {r['BalAcc_mean']:.4f} ± {r['BalAcc_std']:.4f}   AUC = {r['AUC_mean']:.4f} ± {r['AUC_std']:.4f}")
summary.to_csv(OUT_DIR / "summary.csv", index=False)

In [ ]:
print("\n" + "="*60)
print("  Done. Outputs in", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*")):
    print(f"    {f.name}")

---
**Done.** Iterative analysis complete.